In [1]:
import pandas as pd
import numpy as np

CSV_PATH = r"D:\UniStuff\BA\BA-Adrian-Preisler\BE\data\DataSets\titanic.csv"

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
display(df.head(10))

Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [2]:
TARGET_VARIABLE = "Survived"

if TARGET_VARIABLE not in df.columns:
    raise ValueError(f"Target '{TARGET_VARIABLE}' not found in dataframe.")

print("Target:", TARGET_VARIABLE)
print("Target preview:")
print(df[TARGET_VARIABLE].head())

Target: Survived
Target preview:
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64


In [3]:
import os
import json
import shutil
import tempfile
import zipfile
from typing import Optional

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    fbeta_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from autogluon.tabular import TabularPredictor

In [4]:
def zip_dir(src_dir: str, zip_path: str) -> None:
    os.makedirs(os.path.dirname(os.path.abspath(zip_path)), exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(src_dir):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, src_dir)
                z.write(full, arcname=rel)


def _safe_train_test_split(df: pd.DataFrame, target: str, test_size: float, seed: int):
    y = df[target]
    stratify = y

    try:
        vc = y.value_counts(dropna=False)
        if vc.min() < 2:
            stratify = None
    except Exception:
        stratify = None

    return train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=stratify,
    )


def infer_problem_type(y: pd.Series) -> str:
    y_nonnull = y.dropna()
    n = len(y_nonnull)

    if n == 0:
        return "classification"

    n_unique = int(y_nonnull.nunique(dropna=True))
    unique_ratio = n_unique / max(1, n)

    is_numeric = pd.api.types.is_numeric_dtype(y_nonnull)

    if is_numeric:
        if n_unique <= 20 and unique_ratio <= 0.05:
            return "classification"
        return "regression"

    return "classification"

In [5]:
problem_type = infer_problem_type(df[TARGET_VARIABLE])

print("Erkannter Problem-Type:", problem_type)
print("Target value counts:")
print(df[TARGET_VARIABLE].value_counts(dropna=False))

Erkannter Problem-Type: classification
Target value counts:
Survived
0    549
1    342
Name: count, dtype: int64


In [6]:
train_full_df, test_df = _safe_train_test_split(df, TARGET_VARIABLE, test_size=0.2, seed=1337)
train_df, val_df = _safe_train_test_split(train_full_df, TARGET_VARIABLE, test_size=0.2, seed=1337)

print("Train Full:", train_full_df.shape)
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train Full: (712, 12)
Train: (569, 12)
Validation: (143, 12)
Test: (179, 12)


In [7]:
PRESET = "best_quality"     # "good_quality" | "high_quality" | "best_quality"
HPO_MODE = "off"            # "off" | "random" | "grid"
TIME_LIMIT_S = 300
SEED = 1337
NUM_TRIALS = 20

ARTIFACT_PATH = None
# ARTIFACT_PATH = r"D:\UniStuff\BA\artifacts\automl_model.zip"

print("PRESET:", PRESET)
print("HPO_MODE:", HPO_MODE)
print("TIME_LIMIT_S:", TIME_LIMIT_S)
print("SEED:", SEED)
print("NUM_TRIALS:", NUM_TRIALS)
print("ARTIFACT_PATH:", ARTIFACT_PATH)

PRESET: best_quality
HPO_MODE: off
TIME_LIMIT_S: 300
SEED: 1337
NUM_TRIALS: 20
ARTIFACT_PATH: None


In [8]:
hyperparameter_tune_kwargs = None

if HPO_MODE == "random":
    hyperparameter_tune_kwargs = {
        "num_trials": int(NUM_TRIALS),
        "searcher": "random",
        "scheduler": "local",
    }
elif HPO_MODE == "grid":
    hyperparameter_tune_kwargs = {
        "num_trials": int(NUM_TRIALS),
        "searcher": "grid",
        "scheduler": "local",
    }
elif HPO_MODE not in ("off", "none", ""):
    raise ValueError('HPO_MODE must be "off", "random", or "grid"')

print("hyperparameter_tune_kwargs:")
print(hyperparameter_tune_kwargs)

hyperparameter_tune_kwargs:
None


In [9]:
tmp_root = tempfile.mkdtemp(prefix="ag_")
ag_path = os.path.join(tmp_root, "predictor")

if problem_type == "regression":
    eval_metric = "mae"
else:
    eval_metric = "accuracy"

predictor = TabularPredictor(
    label=TARGET_VARIABLE,
    problem_type="regression" if problem_type == "regression" else None,
    eval_metric=eval_metric,
    path=ag_path,
    verbosity=0,
)

print("Temporary predictor path:", ag_path)
print("Eval metric:", eval_metric)

Temporary predictor path: C:\Users\Adri1\AppData\Local\Temp\ag_q8bhsqqi\predictor
Eval metric: accuracy


In [10]:
fit_kwargs = dict(
    train_data=train_df,
    tuning_data=val_df,
    presets=PRESET,
    time_limit=int(TIME_LIMIT_S),
    verbosity=0,
    use_bag_holdout=True,
)

if hyperparameter_tune_kwargs is not None:
    fit_kwargs["hyperparameter_tune_kwargs"] = hyperparameter_tune_kwargs

print("fit_kwargs keys:")
print(list(fit_kwargs.keys()))

fit_kwargs keys:
['train_data', 'tuning_data', 'presets', 'time_limit', 'verbosity', 'use_bag_holdout']


In [11]:
predictor = predictor.fit(**fit_kwargs)

print("Training abgeschlossen.")

C:\Users\Adri1\miniconda3\envs\notebook\Lib\site-packages\sklearn\compose\_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\Adri1\miniconda3\envs\notebook\Lib\site-packages\sklearn\compose\_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\Adri1\miniconda3\envs\notebook\Lib\site-packages\sklearn\compose\_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\Adri1\miniconda3\envs\notebook\Lib\site-packages\sklearn\compose\_column_transformer.py:975: FutureWarning: The parameter `forc

Training abgeschlossen.


In [12]:
y_true = test_df[TARGET_VARIABLE]
X_test = test_df.drop(columns=[TARGET_VARIABLE])

y_pred = predictor.predict(X_test)

print("Prediction preview:")
print(pd.Series(y_pred).head())

Prediction preview:
367    1
861    0
768    0
624    0
642    0
Name: Survived, dtype: int64


In [13]:
lb = predictor.leaderboard(silent=True)
best_model_name = None

if lb is not None and len(lb) > 0 and "model" in lb.columns:
    best_model_name = str(lb.iloc[0]["model"])

print("Best model name:", best_model_name)
display(lb.head(10) if lb is not None else None)

Best model name: NeuralNetTorch_BAG_L1


,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,NeuralNetTorch_BAG_L1,0.846154,accuracy,0.235554,31.280748,0.235554,31.280748,1,True,10
1,WeightedEnsemble_L2,0.846154,accuracy,0.236548,31.320228,0.000993,0.039480,2,True,14
2,RandomForestGini_BAG_L1,0.832168,accuracy,0.073323,0.725619,0.073323,0.725619,1,True,3
3,NeuralNetFastAI_BAG_L1,0.832168,accuracy,9.712834,6.359896,9.712834,6.359896,1,True,8
4,NeuralNetTorch_r79_BAG_L1,0.825175,accuracy,0.267447,9.132121,0.267447,9.132121,1,True,13
5,RandomForestEntr_BAG_L1,0.818182,accuracy,0.068932,0.725312,0.068932,0.725312,1,True,4
6,ExtraTreesEntr_BAG_L1,0.818182,accuracy,0.085469,0.679248,0.085469,0.679248,1,True,7
7,CatBoost_r177_BAG_L1,0.818182,accuracy,0.087208,75.895878,0.087208,75.895878,1,True,12
8,ExtraTreesGini_BAG_L1,0.818182,accuracy,0.096929,0.660495,0.096929,0.660495,1,True,6
9,LightGBMXT_BAG_L1,0.811189,accuracy,0.173600,6.226242,0.173600,6.226242,1,True,1


In [14]:
if ARTIFACT_PATH:
    zip_dir(ag_path, ARTIFACT_PATH)
    print("Artefakt gespeichert:", ARTIFACT_PATH)
else:
    print("Kein Artefaktpfad gesetzt, ZIP wurde nicht erzeugt.")

Kein Artefaktpfad gesetzt, ZIP wurde nicht erzeugt.


In [15]:
if problem_type == "regression":
    y_true_num = pd.to_numeric(y_true, errors="coerce")
    y_pred_num = pd.to_numeric(pd.Series(y_pred), errors="coerce")

    mask = (~y_true_num.isna()) & (~y_pred_num.isna())
    yt = y_true_num[mask].to_numpy()
    yp = y_pred_num[mask].to_numpy()

    mae = float(mean_absolute_error(yt, yp)) if len(yt) else float("nan")
    rmse = float(np.sqrt(mean_squared_error(yt, yp))) if len(yt) else float("nan")
    r2 = float(r2_score(yt, yp)) if len(yt) else float("nan")

    print("AUTOGLUON")
    print("Problem-Type: regression")
    print(f"Test MAE:  {mae:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test R2:   {r2:.4f}")

else:
    acc = float(accuracy_score(y_true, y_pred))
    f1m = float(f1_score(y_true, y_pred, average="macro"))
    f2m = float(fbeta_score(y_true, y_pred, beta=2, average="micro"))

    print("AUTOGLUON")
    print("Problem-Type: classification")
    print(f"Test Acc:       {acc:.4f}")
    print(f"Test F1(macro): {f1m:.4f}")
    print(f"Test F2(micro): {f2m:.4f}")

AUTOGLUON
Problem-Type: classification
Test Acc:       0.7989
Test F1(macro): 0.7840
Test F2(micro): 0.7989


In [16]:
if problem_type == "regression":
    result = {
        "selected_model": "AutoML (AutoGluon)",
        "model_key": "automl",
        "problem_type": "regression",
        "preset": PRESET,
        "hpo_mode": "off" if HPO_MODE in ("off", "none", "") else HPO_MODE,
        "time_limit_s": int(TIME_LIMIT_S),
        "seed": int(SEED),
        "best_model_name": best_model_name,
        "metrics": {
            "test_mae": mae,
            "test_rmse": rmse,
            "test_r2": r2,
        },
    }
else:
    result = {
        "selected_model": "AutoML (AutoGluon)",
        "model_key": "automl",
        "problem_type": "classification",
        "preset": PRESET,
        "hpo_mode": "off" if HPO_MODE in ("off", "none", "") else HPO_MODE,
        "time_limit_s": int(TIME_LIMIT_S),
        "seed": int(SEED),
        "best_model_name": best_model_name,
        "metrics": {
            "test_accuracy": acc,
            "test_f1_macro": f1m,
            "test_f2_micro": f2m,
        },
    }

print(result)

{'selected_model': 'AutoML (AutoGluon)', 'model_key': 'automl', 'problem_type': 'classification', 'preset': 'best_quality', 'hpo_mode': 'off', 'time_limit_s': 300, 'seed': 1337, 'best_model_name': 'NeuralNetTorch_BAG_L1', 'metrics': {'test_accuracy': 0.7988826815642458, 'test_f1_macro': 0.7839903459372486, 'test_f2_micro': 0.7988826815642458}}


In [17]:
shutil.rmtree(tmp_root, ignore_errors=True)
print("Temporary directory entfernt:", tmp_root)

Temporary directory entfernt: C:\Users\Adri1\AppData\Local\Temp\ag_q8bhsqqi


In [18]:
import os
import json
import shutil
import tempfile
import zipfile
from typing import Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    fbeta_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from autogluon.tabular import TabularPredictor


def zip_dir(src_dir: str, zip_path: str) -> None:
    os.makedirs(os.path.dirname(os.path.abspath(zip_path)), exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(src_dir):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, src_dir)
                z.write(full, arcname=rel)


def _safe_train_test_split(df: pd.DataFrame, target: str, test_size: float, seed: int):
    y = df[target]
    stratify = y

    try:
        vc = y.value_counts(dropna=False)
        if vc.min() < 2:
            stratify = None
    except Exception:
        stratify = None

    return train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=stratify,
    )


def infer_problem_type(y: pd.Series) -> str:
    y_nonnull = y.dropna()
    n = len(y_nonnull)
    if n == 0:
        return "classification"

    n_unique = int(y_nonnull.nunique(dropna=True))
    unique_ratio = n_unique / max(1, n)

    is_numeric = pd.api.types.is_numeric_dtype(y_nonnull)

    if is_numeric:
        if n_unique <= 20 and unique_ratio <= 0.05:
            return "classification"
        return "regression"

    return "classification"


def train_autogluon(
    df: pd.DataFrame,
    target_variable: str,
    preset: str = "best_quality",
    hpo_mode: str = "off",
    time_limit_s: int = 300,
    seed: int = 1337,
    num_trials: int = 20,
    artifact_path: Optional[str] = None,
):

    TARGET = target_variable
    if TARGET not in df.columns:
        raise ValueError(f"Target '{TARGET}' not in df columns.")

    problem_type = infer_problem_type(df[TARGET])

    train_full_df, test_df = _safe_train_test_split(df, TARGET, test_size=0.2, seed=seed)
    train_df, val_df = _safe_train_test_split(train_full_df, TARGET, test_size=0.2, seed=seed)

    tmp_root = tempfile.mkdtemp(prefix="ag_")
    ag_path = os.path.join(tmp_root, "predictor")

    try:
        hyperparameter_tune_kwargs = None
        if hpo_mode == "random":
            hyperparameter_tune_kwargs = {
                "num_trials": int(num_trials),
                "searcher": "random",
                "scheduler": "local",
            }
        elif hpo_mode == "grid":
            hyperparameter_tune_kwargs = {
                "num_trials": int(num_trials),
                "searcher": "grid",
                "scheduler": "local",
            }
        elif hpo_mode not in ("off", "none", ""):
            raise ValueError('hpo_mode must be "off", "random", or "grid"')

        if problem_type == "regression":
            eval_metric = "mae"
        else:
            eval_metric = "accuracy"

        predictor = TabularPredictor(
            label=TARGET,
            problem_type="regression" if problem_type == "regression" else None,
            eval_metric=eval_metric,
            path=ag_path,
            verbosity=0,
        )

        fit_kwargs = dict(
            train_data=train_df,
            tuning_data=val_df,
            presets=preset,
            time_limit=int(time_limit_s),
            verbosity=0,
            use_bag_holdout=True,
        )

        if hyperparameter_tune_kwargs is not None:
            fit_kwargs["hyperparameter_tune_kwargs"] = hyperparameter_tune_kwargs

        predictor = predictor.fit(**fit_kwargs)

        y_true = test_df[TARGET]
        X_test = test_df.drop(columns=[TARGET])
        y_pred = predictor.predict(X_test)

        lb = predictor.leaderboard(silent=True)
        best_model_name = None
        if lb is not None and len(lb) > 0 and "model" in lb.columns:
            best_model_name = str(lb.iloc[0]["model"])

        if artifact_path:
            zip_dir(ag_path, artifact_path)

        if problem_type == "regression":
            y_true_num = pd.to_numeric(y_true, errors="coerce")
            y_pred_num = pd.to_numeric(pd.Series(y_pred), errors="coerce")
            mask = (~y_true_num.isna()) & (~y_pred_num.isna())
            yt = y_true_num[mask].to_numpy()
            yp = y_pred_num[mask].to_numpy()

            mae = float(mean_absolute_error(yt, yp)) if len(yt) else float("nan")
            rmse = float(np.sqrt(mean_squared_error(yt, yp))) if len(yt) else float("nan")
            r2 = float(r2_score(yt, yp)) if len(yt) else float("nan")

            return {
                "selected_model": "AutoML (AutoGluon)",
                "model_key": "automl",
                "problem_type": "regression",
                "preset": preset,
                "hpo_mode": "off" if hpo_mode in ("off", "none", "") else hpo_mode,
                "time_limit_s": int(time_limit_s),
                "seed": int(seed),
                "best_model_name": best_model_name,
                "metrics": {
                    "test_mae": mae,
                    "test_rmse": rmse,
                    "test_r2": r2,
                },
            }

        else:
            acc = float(accuracy_score(y_true, y_pred))
            f1m = float(f1_score(y_true, y_pred, average="macro"))
            f2m = float(fbeta_score(y_true, y_pred, beta=2, average="micro"))

            return {
                "selected_model": "AutoML (AutoGluon)",
                "model_key": "automl",
                "problem_type": "classification",
                "preset": preset,
                "hpo_mode": "off" if hpo_mode in ("off", "none", "") else hpo_mode,
                "time_limit_s": int(time_limit_s),
                "seed": int(seed),
                "best_model_name": best_model_name,
                "metrics": {
                    "test_accuracy": acc,
                    "test_f1_macro": f1m,
                    "test_f2_micro": f2m,
                },
            }

    finally:
        shutil.rmtree(tmp_root, ignore_errors=True)

In [19]:
import argparse

p = argparse.ArgumentParser()
p.add_argument("--data", required=True)
p.add_argument("--label", required=True)
p.add_argument("--preset", default="best_quality")
p.add_argument("--hpo_mode", default="off")
p.add_argument("--time_limit_s", type=int, default=300)
p.add_argument("--seed", type=int, default=1337)
p.add_argument("--num_trials", type=int, default=20)
p.add_argument("--out_json", required=True)
p.add_argument("--artifact_path", default=None)
args = p.parse_args()

df = pd.read_csv(args.data)

result = train_autogluon(
    df=df,
    target_variable=args.label,
    preset=args.preset,
    hpo_mode=args.hpo_mode,
    time_limit_s=args.time_limit_s,
    seed=args.seed,
    num_trials=args.num_trials,
    artifact_path=args.artifact_path,
)

with open(args.out_json, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

usage: ipykernel_launcher.py [-h] --data DATA --label LABEL [--preset PRESET] [--hpo_mode HPO_MODE]
                             [--time_limit_s TIME_LIMIT_S] [--seed SEED] [--num_trials NUM_TRIALS] --out_json OUT_JSON
                             [--artifact_path ARTIFACT_PATH]
ipykernel_launcher.py: error: the following arguments are required: --data, --label, --out_json


SystemExit: 2

C:\Users\Adri1\miniconda3\envs\notebook\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
